# Inspect the workspace

Read-only inspection. A fresh clone may have no local assets; UNCONFIGURED and MISSING are expected availability findings, not successful runtime validation.


## Locate the clone and load shared definitions


In [ ]:
from pathlib import Path
import os
_candidate = Path(os.environ.get('TRACE_LAB_ROOT', Path.cwd())).expanduser().resolve()
_candidates = [_candidate] if os.environ.get('TRACE_LAB_ROOT') else [_candidate, *_candidate.parents]
ROOT = next((p for p in _candidates if (p / '.trace-lab-root').is_file() and (p / 'configs').is_dir()), None)
if ROOT is None:
    raise RuntimeError('Open inside trace-lab or set TRACE_LAB_ROOT to the clone')
# Run definition cells in this fresh kernel; this notebook has no execution side effects.
get_ipython().run_line_magic('run', '"' + str(ROOT / 'notebooks/library/configuration.ipynb') + '"')
ROOT = workspace_root(ROOT)


## Resolve configuration and report asset availability


In [ ]:
settings = execution_settings(ROOT)
assets = load_assets(ROOT)
print('Workspace:', ROOT)
print('Configuration:', assets['config_path'])
print(json.dumps({'execution': settings, 'assets': asset_status(assets)}, indent=2))


## Check the preserved environment records


In [ ]:
manifest = json.loads((ROOT / 'environment/runtime/provenance.json').read_text())
for record in manifest['records']:
    assert file_sha256(ROOT / 'environment/runtime' / record['record']) == record['repository_sha256'], record['record']
print(json.dumps({'status': 'PASS', 'environment_records_verified': len(manifest['records']), 'runtime_packages': manifest['python_distributions'], 'scope': 'Workspace and recorded bytes only; no simulator or runtime launched.'}, indent=2))
